In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
try:
    invoices = pd.read_excel("CRATES-TEST-DATA\\SEPTEMBER\\SEPTEMBER-CRATES-23.xlsx", sheet_name="Hansa Report")
    inv = pd.ExcelFile("CRATES-TEST-DATA\\SEPTEMBER\\SEPTEMBER-CRATES-23.xlsx")
except:
    invoices = pd.DataFrame()
    print("Invoice file not found")

In [3]:
try:
    credit_notes = pd.read_excel("CRATES-TEST-DATA\\SEPTEMBER\\SEPTEMBER-CRATES-23.xlsx", sheet_name="Sheet1")
except:
    credit_notes = pd.DataFrame()
    print("Credit note file not found")

In [4]:
invoices.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [5]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18832 entries, 0 to 18831
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        18830 non-null  datetime64[ns]
 1   STATUS       18830 non-null  object        
 2   A/C          18830 non-null  float64       
 3   Invoice No.  18830 non-null  float64       
 4   QTY          18830 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 735.8+ KB


In [6]:
numeric_columns = ['QTY', 'Invoice No.']

In [7]:
# Helper function: Remove only one matching invoice per credit note (not all duplicates)
def remove_one_invoice_per_credit(invoices, credit_notes):
    invoices = invoices.copy()
    credit_notes = credit_notes.copy()
    invoices['is_credit'] = False
    used_credit_idx = set()
    for inv_idx, invoice_row in invoices.iterrows():
        ac = invoice_row['A/C']
        qty = invoice_row['QTY']
        # Only consider unused credit notes for this A/C and QTY
        credit_notes_ac = credit_notes[(credit_notes['A/C'] == ac) & (credit_notes['QTY'] == qty) & (~credit_notes.index.isin(used_credit_idx))]
        for cred_idx, credit_row in credit_notes_ac.iterrows():
            invoices.at[inv_idx, 'is_credit'] = True
            used_credit_idx.add(cred_idx)
            break
    return invoices

In [8]:
# Remove only one matching invoice per credit note (not all duplicates)
invoices = remove_one_invoice_per_credit(invoices, credit_notes)

# Filter out matching invoice rows
filtered_df = invoices[~invoices['is_credit']].copy()
# Drop the helper column
filtered_df = filtered_df.drop(columns=['is_credit'])

# Group by DATES and A/C, summing numeric columns
final_df = filtered_df.groupby(['Invoice No.', 'A/C'], as_index=False)[['QTY']].sum()

In [9]:
invoices.head()

,DATES,STATUS,A/C,Invoice No.,QTY,is_credit
0,NaT,NaN,NaN,NaN,NaN,False
1,2025-09-01,Inv,704.0,704250816.0,1.0,True
2,2025-09-01,Inv,718.0,718250624.0,1.0,True
3,2025-09-01,Inv,859.0,859250492.0,1.0,True
4,2025-09-01,Inv,896.0,896250613.0,1.0,True


In [10]:
invoices.tail()

,DATES,STATUS,A/C,Invoice No.,QTY,is_credit
18827,2025-09-13,Inv,6387.0,638725476.0,1.0,False
18828,2025-09-19,Inv,6387.0,638725478.0,8.0,False
18829,2025-09-20,Inv,6387.0,638725479.0,1.0,False
18830,2025-09-20,Inv,6387.0,638725480.0,1.0,False
18831,2025-09-23,Inv,6387.0,638725481.0,7.0,False


In [11]:
invoices.isna().sum()

DATES          2
STATUS         2
A/C            2
Invoice No.    2
QTY            2
is_credit      0
dtype: int64

In [12]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18832 entries, 0 to 18831
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        18830 non-null  datetime64[ns]
 1   STATUS       18830 non-null  object        
 2   A/C          18830 non-null  float64       
 3   Invoice No.  18830 non-null  float64       
 4   QTY          18830 non-null  float64       
 5   is_credit    18832 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(3), object(1)
memory usage: 754.1+ KB


In [13]:
invoices.describe()

,DATES,A/C,Invoice No.,QTY
count,18830,18830.000000,1.883000e+04,18830.000000
mean,2025-09-11 22:37:06.149761024,2134.736219,1.419174e+09,22.624801
min,2025-09-01 00:00:00,55.000000,5.525028e+07,1.000000
25%,2025-09-06 00:00:00,1266.000000,8.682503e+08,5.000000
50%,2025-09-12 00:00:00,1752.000000,1.543251e+09,11.000000
75%,2025-09-18 00:00:00,2163.000000,1.961250e+09,27.000000
max,2025-09-23 00:00:00,7098.000000,2.298250e+09,505.000000
std,NaN,1581.262048,6.084353e+08,36.596650


In [14]:
invoices.shape

(18832, 6)

In [15]:
credit_notes.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2025-09-01,CN,704,704250827,1
1,2025-09-01,CN,704,704250835,1
2,2025-09-01,CN,708,708250318,1
3,2025-09-01,CN,718,718250627,1
4,2025-09-01,CN,735,735250690,1


In [16]:
credit_notes.tail()

,DATES,STATUS,A/C,Invoice No.,QTY
3338,2025-09-23,CN,1346,1346250307,40
3339,2025-09-23,CN,2194,2194250302,48
3340,2025-09-23,CN,1624,1624250287,50
3341,2025-09-23,CN,2224,2224250285,51
3342,2025-09-23,CN,989,989250286,56


In [17]:
credit_notes.shape

(3343, 5)

In [18]:
credit_notes.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [19]:
credit_notes.isna().sum()

DATES          0
STATUS         0
A/C            0
Invoice No.    0
QTY            0
dtype: int64

In [20]:
credit_notes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3343 entries, 0 to 3342
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        3343 non-null   datetime64[ns]
 1   STATUS       3343 non-null   object        
 2   A/C          3343 non-null   int64         
 3   Invoice No.  3343 non-null   int64         
 4   QTY          3343 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 130.7+ KB


In [21]:
credit_notes.describe()

,DATES,A/C,Invoice No.,QTY
count,3343,3343.000000,3.343000e+03,3343.000000
mean,2025-09-11 21:52:29.865390336,1889.232127,1.550571e+09,3.277595
min,2025-09-01 00:00:00,55.000000,5.525028e+07,1.000000
25%,2025-09-06 00:00:00,1295.000000,1.198251e+09,1.000000
50%,2025-09-12 00:00:00,1728.000000,1.676251e+09,1.000000
75%,2025-09-18 00:00:00,2036.000000,1.972251e+09,2.000000
max,2025-09-23 00:00:00,7098.000000,9.632504e+09,398.000000
std,NaN,1186.286870,5.298000e+08,10.022721


In [22]:
numeric_columns = ['QTY', 'Invoice No.', 'A/C']

In [23]:
credit_notes.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [24]:
# # Helper function: Remove only one matching invoice per credit note (not all duplicates)
# def remove_one_invoice_per_credit(invoices, credit_notes, numeric_columns):
#     invoices = invoices.copy()
#     credit_notes = credit_notes.copy()
#     invoices['is_credit'] = False
#     used_credit_idx = set()
#     for inv_idx, invoice_row in invoices.iterrows():
#         ac = invoice_row['A/C']
#         # Only consider unused credit notes for this A/C
#         credit_notes_ac = credit_notes[(credit_notes['A/C'] == ac) & (~credit_notes.index.isin(used_credit_idx))]
#         for cred_idx, credit_row in credit_notes_ac.iterrows():
#             if all(invoice_row[col] == -credit_row[col] for col in numeric_columns):
#                 invoices.at[inv_idx, 'is_credit'] = True
#                 used_credit_idx.add(cred_idx)
#                 break5
#     return invoices

In [25]:
# Function to check if invoice row matches any credit note row for the same A/C
def is_matching_credit_row(invoice_row, credit_notes, numeric_columns):
    ac = invoice_row['A/C']
    # Filter credit notes for the same A/C
    credit_notes_ac = credit_notes[credit_notes['A/C'] == ac]
    if credit_notes_ac.empty:
        return False
    # Compare numeric columns, accounting for negative values in credit notes
    for _, credit_row in credit_notes_ac.iterrows():
        if all(invoice_row[col] == -credit_row[col] for col in numeric_columns):
            return True
    return False

In [26]:
# Process data datewise and aggregate results (remove only one matching invoice per credit note per day)
all_results = []
if not invoices.empty:
    unique_dates = invoices['DATES'].dropna().unique()
    unique_dates = sorted(unique_dates)
    for date in unique_dates:
        # Filter data for the current date
        daily_invoices = invoices[invoices['DATES'] == date].copy()
        if not credit_notes.empty:
            daily_credit_notes = credit_notes[credit_notes['DATES'] == date].copy()
        else:
            daily_credit_notes = pd.DataFrame()

        # Remove only one matching invoice per credit note for the day
        if not daily_credit_notes.empty:
            daily_invoices = remove_one_invoice_per_credit(daily_invoices, daily_credit_notes)
        else:
            daily_invoices['is_credit'] = False

        # Filter out matching invoice rows
        filtered_df = daily_invoices[~daily_invoices['is_credit']].copy()
        # Drop the helper column
        filtered_df = filtered_df.drop(columns=['is_credit'])

        # Group by DATES and A/C, summing QTY and keeping one representative invoice number
        if not filtered_df.empty:
            # Group by DATES and A/C, sum QTY, and keep the first invoice number for each group
            daily_result = filtered_df.groupby(['DATES', 'A/C']).agg({
                'QTY': 'sum',
                'Invoice No.': 'first',  # Keep the first invoice number from each group
                'STATUS': 'first'  # Keep the first status from each group
            }).reset_index()
            
            all_results.append(daily_result)

# Combine all daily results
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df = final_df.sort_values(by=['DATES', 'A/C'])
    # Reorder columns to match expected format
    final_df = final_df[['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY']]
else:
    final_df = pd.DataFrame()

# Display the final combined result
print("Final processed data:")
print(final_df)

Final processed data:
           DATES STATUS     A/C  Invoice No.    QTY
0     2025-09-01    Inv    55.0   55250282.0   38.0
1     2025-09-01    Inv    57.0   57250295.0  206.0
2     2025-09-01    Inv    62.0   62250313.0  134.0
3     2025-09-01    Inv    63.0   63250279.0  119.0
4     2025-09-01    Inv    82.0   82250307.0   54.0
...          ...    ...     ...          ...    ...
16069 2025-09-23    Inv  7064.0  706425345.0   10.0
16070 2025-09-23    Inv  7085.0  708525393.0    8.0
16071 2025-09-23    Inv  7094.0  709425394.0    3.0
16072 2025-09-23    Inv  7096.0  709625272.0    4.0
16073 2025-09-23    Inv  7098.0  709825440.0    2.0

[16074 rows x 5 columns]


In [27]:
final_df = final_df.to_excel('CRATES-OUTPUT-DATA\\SEPTEMBER\\SEPTEMBER-CRATES-1.xlsx', index=False)